In [2]:
# !pip install -q transformers datasets peft accelerate evaluate sacrebleu sentencepiece bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 10.6 MB/s eta 0:00:00


In [1]:
import torch
import glob
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
import evaluate

In [2]:
# Load and Combine XLSX Files

xlsx_files = glob.glob("*.xlsx")
processed_dfs = []

for file in xlsx_files:
    df = pd.read_excel(file)
    df.columns = df.columns.str.strip()

    # Identify dialect and standard columns dynamically per file
    dialect_cols = [c for c in df.columns if 'dialect' in c.lower()]
    standard_cols = [c for c in df.columns if 'standard' in c.lower()]

    if dialect_cols and standard_cols:
      d_col = dialect_cols[0]
      s_col = standard_cols[0]

      temp_df = pd.DataFrame()
      temp_df['source'] = df[d_col].astype(str).str.strip()
      temp_df['target'] = df[s_col].astype(str).str.strip()

      temp_df['dialect_type'] = d_col.replace("Dialect", '').strip()

      temp_df = temp_df.dropna()
      temp_df = temp_df[(temp_df['source'] != '') & (temp_df['target'] != '') & (temp_df['source'] != 'nan')]

      processed_dfs.append(temp_df)

full_df = pd.concat(processed_dfs, ignore_index=True)
print(f"Dataset successfully loaded with {len(full_df)} total rows.")
print("Distribution per dialect:\n", full_df['dialect_type'].value_counts())




Dataset successfully loaded with 17329 total rows.
Distribution per dialect:
 dialect_type
Sylheti         9320
Chittagonian    8009
Name: count, dtype: int64


In [3]:
train_df, temp_df = train_test_split(
    full_df,
    test_size=0.20,
    random_state=42,
    stratify=full_df['dialect_type']
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df['dialect_type']
)

dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df[['source', 'target', 'dialect_type']], preserve_index=False),
    'validation': Dataset.from_pandas(val_df[['source', 'target', 'dialect_type']], preserve_index=False),
    'test': Dataset.from_pandas(test_df[['source', 'target', 'dialect_type']], preserve_index=False)
})

print("\nDataset splits created successfully:")
print(dataset)


Dataset splits created successfully:
DatasetDict({
    train: Dataset({
        features: ['source', 'target', 'dialect_type'],
        num_rows: 13863
    })
    validation: Dataset({
        features: ['source', 'target', 'dialect_type'],
        num_rows: 1733
    })
    test: Dataset({
        features: ['source', 'target', 'dialect_type'],
        num_rows: 1733
    })
})


In [4]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"
SRC_LANG = "ben_Beng"  # Bengali script source
TGT_LANG = "ben_Beng"  # Standard Bengali target

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    src_lang=SRC_LANG,
    tgt_lang=TGT_LANG
)

In [5]:
# 8-bit model
if torch.cuda.is_available():
    quantization_config = BitsAndBytesConfig(load_in_8bit=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        device_map={"": 0}   # single GPU
    )
else:
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)



model = prepare_model_for_kbit_training(model)

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

In [6]:
# LoRA
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"]
)

model = get_peft_model(model, peft_config=peft_config)
model.print_trainable_parameters()
model.config.use_cache = False

trainable params: 4,718,592 || all params: 619,792,384 || trainable%: 0.7613


In [7]:
# tokenization

max_input_length = 128
max_target_length = 128

def preprocess_function(examples):
    inputs = examples['source']
    targets = examples['target']

    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)
    labels = tokenizer(text_target=targets, max_length=max_target_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/13863 [00:00<?, ? examples/s]

Map:   0%|          | 0/1733 [00:00<?, ? examples/s]

Map:   0%|          | 0/1733 [00:00<?, ? examples/s]

In [8]:
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Replace -100 in labels (used for padding) before decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [[l.strip()] for l in decoded_labels]  # sacrebleu wants list-of-lists

    bleu = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
    chrf = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {
        "bleu": bleu["score"],
        "chrf": chrf["score"],
    }

In [11]:
import warnings
import logging
import torch
import logging

import os
import ctypes
import warnings
import logging

#  Disable bitsandbytes internal C++ logging
os.environ["BITSANDBYTES_NOWELCOME"] = "1"

try:
    import bitsandbytes as bnb
    # Mute the bitsandbytes autograd warnings module directly
    bnb.autograd._functions.logger.setLevel(logging.ERROR)
except Exception:
    pass

#  Suppress Python-level warning triggers
warnings.filterwarnings("ignore", message=".*MatMul8bitLt.*")
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("transformers").setLevel(logging.ERROR)



# Save trained adapter weights locally
model.save_pretrained("./final_nllb_dialect_lora")
tokenizer.save_pretrained("./final_nllb_dialect_lora")

# Silence bitsandbytes warnings
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)

# Silence transformers warnings completely
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()


# 1. Suppress bitsandbytes output spam inside the active runtime scope
warnings.filterwarnings("ignore", message=".*MatMul8bitLt.*")
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("transformers").setLevel(logging.ERROR)




# training arguments
# training_args = Seq2SeqTrainingArguments(
#     output_dir="./nllb_dialect_adapter",
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     learning_rate=2e-4,
#     per_device_train_batch_size=16,   # reduce for debugging
#     per_device_eval_batch_size=16,
#     weight_decay=0.01,
#     save_total_limit=2,
#     num_train_epochs=5,              # reduce for debugging
#     predict_with_generate=True,     # disable for debugging
#     fp16=torch.cuda.is_available(),
#     logging_steps=1,                 # log every step
#     report_to="none"
# )




# data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# bleu_metric = evaluate.load("sacrebleu")
# chrf_metric = evaluate.load("chrf")

# trainer = Seq2SeqTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_datasets["train"],
#     eval_dataset=tokenized_datasets["validation"],
#     processing_class=tokenizer,
#     data_collator=data_collator,
#     compute_metrics=compute_metrics,   # <-- new
# )
# # Execute Fine-Tuning
# trainer.train()

# # Save trained adapter weights locally
# model.save_pretrained("./final_nllb_dialect_lora")
# tokenizer.save_pretrained("./final_nllb_dialect_lora")






training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb_dialect_adapter",
    eval_strategy="epoch",            # Evaluate once at epoch end
    save_strategy="epoch",            # Save once at epoch end
    learning_rate=5e-4,
    per_device_train_batch_size=16,   # Increased from 1 to maximize T4 GPU VRAM utilization
    per_device_eval_batch_size=16,    # Process validation set in larger parallel batches
    gradient_accumulation_steps=1,    # Direct updates
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=5,
    predict_with_generate=False,      # FAST EVALUATION: Calculates loss without slow autoregressive token generation
    fp16=True,                        # Mixed precision on T4 CUDA cores
    logging_steps=20,                 # Log metrics every 20 steps instead of every 1
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    # Note: compute_metrics is omitted here because predict_with_generate=False
    # computes fast CrossEntropy loss during validation. You will run BLEU/chrF on test set after training.
)

# Clear unused VRAM memory
torch.cuda.empty_cache()

# Execute Fine-Tuning
trainer.train()

# Save trained adapter weights locally
model.save_pretrained("./final_nllb_dialect_lora")
tokenizer.save_pretrained("./final_nllb_dialect_lora")

{'loss': '2.227', 'grad_norm': '0.5431', 'learning_rate': '0.0004978', 'epoch': '0.02307'}
{'loss': '1.472', 'grad_norm': '0.419', 'learning_rate': '0.0004955', 'epoch': '0.04614'}
{'loss': '1.349', 'grad_norm': '0.5221', 'learning_rate': '0.0004932', 'epoch': '0.0692'}
{'loss': '1.277', 'grad_norm': '0.5321', 'learning_rate': '0.0004909', 'epoch': '0.09227'}
{'loss': '1.219', 'grad_norm': '0.5096', 'learning_rate': '0.0004886', 'epoch': '0.1153'}
{'loss': '1.16', 'grad_norm': '1.405', 'learning_rate': '0.0004863', 'epoch': '0.1384'}
{'loss': '1.182', 'grad_norm': '0.5899', 'learning_rate': '0.000484', 'epoch': '0.1615'}
{'loss': '1.153', 'grad_norm': '0.5745', 'learning_rate': '0.0004817', 'epoch': '0.1845'}
{'loss': '1.087', 'grad_norm': '0.4357', 'learning_rate': '0.0004794', 'epoch': '0.2076'}
{'loss': '1.077', 'grad_norm': '1.472', 'learning_rate': '0.000477', 'epoch': '0.2307'}
{'loss': '1.088', 'grad_norm': '0.7714', 'learning_rate': '0.0004747', 'epoch': '0.2537'}
{'loss': '1.0

('./final_nllb_dialect_lora/tokenizer_config.json',
 './final_nllb_dialect_lora/tokenizer.json')

In [13]:
import pandas as pd
import torch
import evaluate

bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

test_sources = dataset["test"]["source"]
test_targets = dataset["test"]["target"]
test_dialects = dataset["test"]["dialect_type"]

# Fetch target language BOS token ID safely for NLLB
target_lang_id = tokenizer.convert_tokens_to_ids(TGT_LANG)

predictions = []
batch_size = 8

model.eval()

# Generate translations for test set
for i in range(0, len(test_sources), batch_size):
    batch_inputs = test_sources[i:i + batch_size]
    inputs = tokenizer(batch_inputs, padding=True, truncation=True, max_length=128, return_tensors="pt").to("cuda")

    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=target_lang_id,
            max_length=128
        )

    decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    predictions.extend(decoded_preds)

# Create DataFrame for results analysis
results_df = pd.DataFrame({
    'dialect': test_dialects,
    'source': test_sources,
    'target': test_targets,
    'prediction': predictions
})

# Overall Scores
references = [[tgt] for tgt in test_targets]
overall_bleu = bleu_metric.compute(predictions=predictions, references=references)
overall_chrf = chrf_metric.compute(predictions=predictions, references=references)

print(f"\n==========================================")
print(f"--- OVERALL TEST RESULTS ---")
print(f"BLEU: {overall_bleu['score']:.2f}")
print(f"chrF: {overall_chrf['score']:.2f}")
print(f"==========================================")

# Breakdown by Dialect
print("\n--- PER-DIALECT BREAKDOWN ---")
for dialect_name in results_df['dialect'].unique():
    subset = results_df[results_df['dialect'] == dialect_name]
    sub_preds = subset['prediction'].tolist()
    sub_refs = [[tgt] for tgt in subset['target'].tolist()]

    sub_bleu = bleu_metric.compute(predictions=sub_preds, references=sub_refs)
    sub_chrf = chrf_metric.compute(predictions=sub_preds, references=sub_refs)

    print(f"\nDialect: {dialect_name} (Samples: {len(subset)})")
    print(f"  BLEU: {sub_bleu['score']:.2f}")
    print(f"  chrF: {sub_chrf['score']:.2f}")

# Sample Predictions
print("\n--- SAMPLE PREDICTIONS ---")
print(results_df[['dialect', 'source', 'prediction', 'target']].head(6).to_string())


--- OVERALL TEST RESULTS ---
BLEU: 47.09
chrF: 74.65

--- PER-DIALECT BREAKDOWN ---

Dialect: Sylheti (Samples: 932)
  BLEU: 42.15
  chrF: 70.94

Dialect: Chittagonian (Samples: 801)
  BLEU: 52.23
  chrF: 78.74

--- SAMPLE PREDICTIONS ---
        dialect                                                                   source                                                                 prediction                                                                   target
0       Sylheti  অংকের ভাষায়— একজন ম্যাসাজম্যান গা টিপ্পা অন্য এখজনরে শারীরিক আরাম দিরো।  অংকের ভাষায় একজন ম্যাসাজম্যান গা টিপ্পা অন্য একজনকে শারীরিক আরাম দিচ্ছে।  অংকের ভাষায়— একজন ম্যাসাজম্যান গা টিপে অন্য একজনকে শারীরিক আরাম দিচ্ছে।
1  Chittagonian                           পৃথিবীর প্রতি আঁর  মমতা আছে বুলিই এই ধর্মপালন।                              পৃথিবীর প্রতি আমার মমতা আছে বলেই এই ধর্মপালন।                            পৃথিবীর প্রতি আমার মমতা আছে বলেই এই ধর্মপালন।
2  Chittagonian                                আঁ